# Salary Prediction — Machine Learning & MLOps

This notebook is based on the supplied `salary-prediction.ipynb` and `Salary_Data(1).csv`.

**Task:** Predict salary from years of experience.

The original notebook explores:
- dataset inspection
- missing values and duplicates
- quartiles and IQR
- correlation
- train/test split
- Linear Regression
- Polynomial Regression
- Support Vector Regression (SVR)
- model comparison
- sample salary prediction

This structured project adds reproducible preprocessing, Decision Tree/Random Forest baseline models, saved artifacts, evaluation outputs, and MLflow tracking.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "salary.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


## 1. Explore the Dataset

In [ ]:
print("First 10 rows:")
display(df.head(10))

print("\nLast 10 rows:")
display(df.tail(10))

print("\nShape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nStatistical summary:")
display(df.describe())


## 2. Data Quality Checks

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Q1 = df.quantile(0.25)
Q2 = df.quantile(0.50)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

print("\nQ1:")
print(Q1)
print("\nMedian (Q2):")
print(Q2)
print("\nQ3:")
print(Q3)
print("\nIQR:")
print(IQR)

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[((df < lower_bound) | (df > upper_bound)).any(axis=1)]

print("\nPotential IQR outliers:")
display(outliers)


## 3. Correlation and Relationship

In [ ]:
correlation = df["YearsExperience"].corr(df["Salary"])
print("Correlation between YearsExperience and Salary:", correlation)

plt.figure(figsize=(8, 5))
plt.scatter(df["YearsExperience"], df["Salary"], alpha=0.8)
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Years of Experience vs Salary")
plt.tight_layout()
plt.show()


There are no categorical columns in the supplied dataset, so categorical encoding is not required.

In [ ]:
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


## 4. Linear Regression

In [ ]:
linear = LinearRegression()
linear.fit(X_train, y_train)
linear_pred = linear.predict(X_test)

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {
        "R2": r2,
        "R2_percent": r2 * 100,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
    }

print("Linear Regression Results")
print(regression_report(y_test, linear_pred))
print(f"Equation: Salary = {linear.intercept_:.2f} + {linear.coef_[0]:.2f} * YearsExperience")

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.8, label="Actual Data")
plt.plot(X, linear.predict(X), label="Regression Line")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Linear Regression")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Polynomial Regression

In [ ]:
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

polynomial = LinearRegression()
polynomial.fit(X_train_poly, y_train)
poly_pred = polynomial.predict(X_test_poly)

print("Polynomial Regression Results")
print(regression_report(y_test, poly_pred))

X_curve = np.linspace(
    X["YearsExperience"].min(),
    X["YearsExperience"].max(),
    200
).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.8, label="Actual Data")
plt.plot(X_curve, polynomial.predict(poly.transform(X_curve)), label="Polynomial Regression")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Polynomial Regression (Degree 2)")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Support Vector Regression (SVR)

In [ ]:
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()

svr = SVR(kernel="rbf", C=100, gamma="scale", epsilon=0.1)
svr.fit(X_train_scaled, y_train_scaled)

svr_pred_scaled = svr.predict(X_test_scaled)
svr_pred = y_scaler.inverse_transform(svr_pred_scaled.reshape(-1, 1)).ravel()

print("SVR Results")
print(regression_report(y_test, svr_pred))


## 7. MLOps Baseline Models — Decision Tree and Random Forest

In [ ]:
# Scale the numerical feature for a consistent saved-array pipeline.
pipeline_scaler = StandardScaler()
X_train_final = pipeline_scaler.fit_transform(X_train)
X_test_final = pipeline_scaler.transform(X_test)

decision_tree = DecisionTreeRegressor(
    max_depth=5, min_samples_leaf=2, random_state=42
)
decision_tree.fit(X_train_final, y_train)

random_forest = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
random_forest.fit(X_train_final, y_train)

dt_pred = decision_tree.predict(X_test_final)
rf_pred = random_forest.predict(X_test_final)

comparison = pd.DataFrame([
    {"Model": "Linear Regression", **regression_report(y_test, linear_pred)},
    {"Model": "Polynomial Regression", **regression_report(y_test, poly_pred)},
    {"Model": "SVR", **regression_report(y_test, svr_pred)},
    {"Model": "Decision Tree", **regression_report(y_test, dt_pred)},
    {"Model": "Random Forest", **regression_report(y_test, rf_pred)},
])

display(comparison)


## 8. Example Prediction

In [ ]:
sample_experience = 10.0
sample = pd.DataFrame({"YearsExperience": [sample_experience]})

linear_salary = linear.predict(sample)[0]
poly_salary = polynomial.predict(poly.transform(sample))[0]

sample_scaled = pipeline_scaler.transform(sample)
rf_salary = random_forest.predict(sample_scaled)[0]

print(f"For {sample_experience} years of experience:")
print(f"Linear Regression     : {linear_salary:,.2f}")
print(f"Polynomial Regression : {poly_salary:,.2f}")
print(f"Random Forest         : {rf_salary:,.2f}")


## 9. Save Core MLOps Artifacts

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
models_dir = PROJECT_ROOT / "models"
outputs_dir = PROJECT_ROOT / "outputs"

processed_dir.mkdir(exist_ok=True, parents=True)
models_dir.mkdir(exist_ok=True, parents=True)
outputs_dir.mkdir(exist_ok=True, parents=True)

np.save(processed_dir / "X_train_final.npy", X_train_final)
np.save(processed_dir / "X_test_final.npy", X_test_final)
np.save(processed_dir / "y_train.npy", y_train.to_numpy())
np.save(processed_dir / "y_test.npy", y_test.to_numpy())

joblib.dump(pipeline_scaler, models_dir / "scaler.pkl")
joblib.dump({"used": False, "encoder": None}, models_dir / "ohe.pkl")
joblib.dump(decision_tree, models_dir / "decision_tree_baseline.pkl")
joblib.dump(random_forest, models_dir / "random_forest_baseline.pkl")
joblib.dump(random_forest, models_dir / "random_forest_mlflow.pkl")

np.save(outputs_dir / "predictions.npy", rf_pred)
np.save(outputs_dir / "salary_predictions.npy", rf_pred)

print("Artifacts saved under data/processed, models and outputs.")


## 10. Final Notes

- The supplied dataset contains one numerical feature and one numerical target.
- No categorical encoding is needed.
- Regression should be evaluated with MAE, MSE, RMSE and R² rather than classification accuracy.
- The structured command-line pipelines in `pipelines/` reproduce the preprocessing, training, evaluation and MLflow tracking steps.